In [3]:
import os
from dotenv import load_dotenv

# 환경 변수 로드 - .env 파일에서 Neo4j 연결 정보(URI, 사용자명, 비밀번호 등)를 불러옵니다.
# 이 과정을 통해 코드에 직접 민감한 정보를 하드코딩하지 않고 안전하게 관리할 수 있습니다.
# .env 파일이 현재 작업 디렉토리에 있어야 하며, 없을 경우 환경 변수가 로드되지 않습니다.
load_dotenv()

True

In [4]:
from langchain_neo4j import Neo4jGraph

# LangChain 도구 활용 - DB 연결 객체 초기화 
# Neo4jGraph 클래스는 LangChain 라이브러리에서 제공하는 Neo4j 그래프 데이터베이스 연결 도구입니다.
# 이 객체를 통해 Cypher 쿼리를 실행하고 그래프 데이터를 조작할 수 있습니다.
# .env 파일에서 로드한 환경 변수를 사용하여 안전하게 연결 정보를 관리합니다.
# - url: Neo4j 데이터베이스 연결 주소 (AuraDB의 경우 neo4j+s:// 프로토콜 사용)
# - username: Neo4j 데이터베이스 접속 사용자 이름 (기본값은 'neo4j')
# - password: Neo4j 데이터베이스 접속 비밀번호
graph = Neo4jGraph( 
    url=os.getenv("NEO4J_URI"), 
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD"),
)

In [7]:
# 테스트 쿼리 실행 - Neo4j 연결이 제대로 작동하는지 확인하기 위한 간단한 테스트
# Cypher 쿼리를 사용하여 Test 레이블을 가진 노드를 생성하고 반환합니다
cypher_query = """
CREATE (n:Test {name: "Hello AuraDB"}) 
RETURN n
"""

# graph.query() 메서드를 사용하여 Cypher 쿼리를 실행합니다
# CREATE 구문은 새 노드를 생성하고, RETURN은 생성된 노드를 결과로 반환합니다
# 이 쿼리가 성공적으로 실행되면 Neo4j 데이터베이스 연결이 정상적으로 작동하는 것입니다
graph.query(cypher_query)

[{'n': {'name': 'Hello AuraDB'}}]

In [6]:
def reset_database(graph):
    """
    데이터베이스 초기화하기
    """
    # 모든 노드와 관계 삭제
    graph.query("MATCH (n) DETACH DELETE n")
    
    # 모든 제약조건 삭제
    constraints = graph.query("SHOW CONSTRAINTS")
    for constraint in constraints:
        constraint_name = constraint.get("name")
        if constraint_name:
            graph.query(f"DROP CONSTRAINT {constraint_name}")
    
    # 모든 인덱스 삭제
    indexes = graph.query("SHOW INDEXES")
    for index in indexes:
        index_name = index.get("name")
        index_type = index.get("type")
        if index_name and index_type != "CONSTRAINT":
            graph.query(f"DROP INDEX {index_name}")
    
    print("데이터베이스가 초기화되었습니다.")

# 데이터베이스 초기화
reset_database(graph)

데이터베이스가 초기화되었습니다.


In [7]:
import os
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph

# --- 1. 환경 변수 로드 ---
# .env 파일에서 Neo4j 연결 정보를 불러옵니다.
load_dotenv()
print(".env 파일에서 환경 변수를 로드했습니다.")

# 환경 변수가 제대로 로드되었는지 간단히 확인
NEO4J_URI = os.getenv("NEO4J_URI")
if not NEO4J_URI:
    print("❌ 오류: .env 파일에서 NEO4J_URI를 찾을 수 없습니다.")
    exit()

print(f"'{NEO4J_URI}'에 연결을 시도합니다...")

try:
    # --- 2. Neo4jGraph 인스턴스(객체) 생성 ---
    # 이 단계에서 LangChain은 DB에 접속을 시도하고 인증을 확인합니다.
    graph = Neo4jGraph(
        url=os.getenv("NEO4J_URI"),
        username=os.getenv("NEO4J_USERNAME"),
        password=os.getenv("NEO4J_PASSWORD"),
    )
    
    # --- 3. 테스트 쿼리 실행 (노드 생성) ---
    print("연결 성공. 테스트 노드를 생성합니다...")
    create_query = """
    CREATE (n:TestNode {name: "LangChainConnectionTest", timestamp: timestamp()})
    RETURN n.name AS name, n.timestamp AS created_at
    """
    result = graph.query(create_query)
    
    print("\n✅ Neo4j AuraDB 연결 및 쓰기 테스트 성공!")
    print(f"성공적으로 생성된 노드 정보: {result}")

    # --- 4. 테스트 데이터 정리 (생성한 노드 삭제) ---
    print("\n테스트 데이터를 정리합니다...")
    delete_query = """
    MATCH (n:TestNode {name: "LangChainConnectionTest"})
    DELETE n
    RETURN count(n) AS deleted_count
    """
    graph.query(delete_query)
    print("테스트 노드를 삭제했습니다. `build_graph.py`를 실행할 준비가 되었습니다.")

except Exception as e:
    # --- 5. 실패 시 오류 메시지 출력 ---
    print("\n❌ Neo4j AuraDB 연결 또는 쿼리 실패 ❌")
    print(f"\n[오류 메시지]\n{e}")
    
    print("\n[체크리스트]")
    print("1. .env 파일의 NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD가 정확한지 다시 확인하세요.")
    print("2. Neo4j AuraDB 대시보드에서 데이터베이스가 'Running' 상태인지 확인하세요.")
    print("3. 인터넷 연결이나 방화벽 설정을 확인하세요.")

finally:
    # Neo4jGraph 객체는 내부적으로 연결 풀(pool)을 관리하므로,
    # 스크립트가 종료될 때 자동으로 연결이 닫힙니다.
    print("\n테스트 스크립트를 종료합니다.")

.env 파일에서 환경 변수를 로드했습니다.
'neo4j+s://69a7530c.databases.neo4j.io'에 연결을 시도합니다...
연결 성공. 테스트 노드를 생성합니다...

✅ Neo4j AuraDB 연결 및 쓰기 테스트 성공!
성공적으로 생성된 노드 정보: [{'name': 'LangChainConnectionTest', 'created_at': 1761036704869}]

테스트 데이터를 정리합니다...
테스트 노드를 삭제했습니다. `build_graph.py`를 실행할 준비가 되었습니다.

테스트 스크립트를 종료합니다.
